# v3 — Simulated Annealing: Search Visualization

How Simulated Annealing moves through the search space on the synthetic
FD model, **with and without free-surface multiples**: the objective of
the current solution along the run, the temperature and the share of
worse moves accepted, the number of picks, snapshots of the solution over
the semblance panel, where the evaluations were spent, and an animation.

Same neighborhood as Hill Climbing and Tabu Search (shift one pick,
insert / remove one pick); only the acceptance rule changes: a worse
neighbor is accepted with probability exp(delta / T), and T decreases
geometrically over the budget, from T0 (set so that losing one pick's
worth of value, tau, is accepted 80% of the time) to T0 / 1000.

Formulation: the same as `configs/batch_fd_v3_sep200.json` (v3, robust
semblance, 200 ms separation, 100,000 evaluations).

**Execution order:**
1. Setup
2. Parameters
3. The data: shot record, CDP gather and semblance panel
4. Load the cases
5. Run on both cases, with the trajectory recorded
6. Objective and number of picks along the run
7. Snapshots of the current solution
8. Where the search looked
9. Temperature and acceptance
10. Animation
11. Summary

## 1. Setup

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Project root: walk up to the folder that contains src/
from pathlib import Path
project_root = Path.cwd().resolve()
while not (project_root / 'src').is_dir():
    if project_root.parent == project_root:
        raise RuntimeError('Project root (folder containing src/) not found.')
    project_root = project_root.parent
project_root = str(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

FIG_DIR = os.path.join(project_root, 'results', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Project root: {project_root}')

In [ ]:
from src.seismic.cases      import load_case
from src.seismic.semblance  import compute_semblance, robust_threshold, ROBUST_SEMBLANCE
from src.analysis.reference import get_reference, match_reference
from src.analysis           import search_viz
from src.optimizers.simulated_annealing import SimulatedAnnealing

print('All modules imported successfully.')

## 2. Parameters

In [ ]:
ALG = 'Simulated Annealing'
TAG = 'sa'

# --- Cases: the synthetic FD model with and without free-surface multiples ---
CASES = ['fd_layered_cdp2500_fs', 'fd_layered_cdp2500_nofs']
LABEL = {'fd_layered_cdp2500_fs':   'with free surface (multiples)',
         'fd_layered_cdp2500_nofs': 'without free surface'}

# --- Formulation: same as configs/batch_fd_v3_sep200.json (v3, robust semblance) ---
VEL_MIN      = 1400.0
VEL_MAX      = 5000.0
VEL_STEP     = 20.0          # panel used for tau and for the figures
MIN_SEP_MS   = 200.0
TAU_FACTOR   = 10.0
P_INSERT     = 0.1
P_REMOVE     = 0.1
SEMBLANCE    = ROBUST_SEMBLANCE   # stretch mute 0.5, energy floor 1%, min fold 10

# --- Budget and seed (the same for both cases) ---
MAX_EVALS    = 100000
SEED         = 42
RECORD_EVERY = 1             # keep one trajectory record every RECORD_EVERY iterations

# --- Local moves ---
STEP_TIME    = 5
STEP_VEL     = 50.0
N_NEIGHBORS  = 30
MAX_ITER     = 3 * MAX_EVALS // N_NEIGHBORS

# --- Figures ---
V_SHOW       = 3000.0        # velocity axis of the panels (m/s)

# --- Simulated Annealing ---
SA_T0          = None        # None: T0 = tau / -ln(SA_P0)
SA_P0          = 0.8         # acceptance of a worse move of size tau at the start
SA_FINAL_RATIO = 1e-3        # final temperature = T0 * SA_FINAL_RATIO
RECORD_EVERY   = 20          # one evaluation per iteration: record every 20

## 3. The data: shot record, CDP gather and semblance panel

The two cases side by side, before any picking: a shot record (all
receivers of the line; the red line marks the source), the CDP gather the
algorithm works on, and its semblance panel (same robust semblance as the
objective). With the free surface, the surface multiples appear in all
three views; without it, only the primaries (and weaker diffractions and
interbed energy) remain.

The shot records are in `data/synthetic_fd/` (not versioned); if they are
missing, run `python scripts/model_fd.py`.

In [ ]:
SHOT = 1                  # shot shown: 0 = near offset, 1 = middle, 2 = far offset

vels_show = np.arange(VEL_MIN, V_SHOW + VEL_STEP, VEL_STEP, dtype=np.float32)
fig, axes = plt.subplots(2, 3, figsize=(18, 13), gridspec_kw={'width_ratios': [1.6, 1, 1]})
for row, name in enumerate(CASES):
    c  = load_case(os.path.join(project_root, 'cases', f'{name}.npz'))
    tg = c['t_grid']

    # Shot record (time gain for display only)
    shot_path = os.path.join(project_root, 'data', 'synthetic_fd', f'{name}_shots.npz')
    ax = axes[row, 0]
    if os.path.exists(shot_path):
        s  = np.load(shot_path)
        li = s['line_idx']; rx = s['rec_x'][li]
        rec = s['shots'][SHOT][li] * (s['t_ms'] / s['t_ms'][-1])[None, :]
        clip = np.percentile(np.abs(rec), 99)
        ax.imshow(rec.T, aspect='auto', cmap='gray', vmin=-clip, vmax=clip,
                  extent=[rx[0], rx[-1], tg[-1], tg[0]])
        ax.axvline(s['shot_x'][SHOT], color='red', lw=0.8)
        ax.set_title(f"Shot at {s['shot_x'][SHOT]:.0f} m — {LABEL[name]}")
    else:
        ax.text(0.5, 0.5, 'shot records not found: run scripts/model_fd.py',
                ha='center', va='center', transform=ax.transAxes)
    ax.set_ylabel('Time (ms)')

    # CDP gather (as the algorithm sees it: direct-wave mute + time gain)
    g = c['gather']; clip = np.percentile(np.abs(g), 99)
    axes[row, 1].imshow(g.T, aspect='auto', cmap='gray', vmin=-clip, vmax=clip,
                        extent=[c['offsets'][0], c['offsets'][-1], tg[-1], tg[0]])
    axes[row, 1].set_title(f"CDP gather at {c['meta']['model']['cdp_x_m']:.0f} m")

    # Semblance panel, no picks
    p = compute_semblance(g, c['offsets'], vels_show, c['dt_ms'], **SEMBLANCE)
    axes[row, 2].imshow(p.T, aspect='auto', cmap='jet', vmin=0, vmax=1,
                        extent=[VEL_MIN, V_SHOW, tg[-1], tg[0]])
    axes[row, 2].set_title('Semblance (robust)')
for ax, lab in zip(axes[1], ('Receiver position (m)', 'Offset (m)', 'Velocity (m/s)')):
    ax.set_xlabel(lab)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'search_data_fs_nofs.png'), dpi=130, bbox_inches='tight')
plt.show()

## 4. Load the cases

For each case: semblance panel and tau with the same options as the
objective, and the reference picks (semblance peak of each expected
event) for the figures — `+` white = primary, `+` magenta = surface
multiple. The algorithm never sees the reference.

In [ ]:
data = {}
for name in CASES:
    case = load_case(os.path.join(project_root, 'cases', f'{name}.npz'))
    velocities = np.arange(VEL_MIN, VEL_MAX + VEL_STEP, VEL_STEP, dtype=np.float32)
    panel = compute_semblance(case['gather'], case['offsets'], velocities, case['dt_ms'], **SEMBLANCE)
    data[name] = {
        'case': case, 'gather': case['gather'], 'offsets': case['offsets'],
        'dt': case['dt_ms'], 't_grid': case['t_grid'], 'panel': panel,
        'tau': robust_threshold(panel, factor=TAU_FACTOR),
        'extent': [VEL_MIN, VEL_MAX, case['t_grid'][-1], case['t_grid'][0]],
        'reference': get_reference(case, **SEMBLANCE),
    }
    print(f"{name:26s} gather {case['gather'].shape}   tau = {data[name]['tau']:.3f}")

## 5. Run on both cases, with the trajectory recorded

`record_trajectory=True` stores the current solution along the run. It
only reads the search state, so each run is identical to the one without
recording (same seed, same result). Same seed and budget on both cases.

In [ ]:
def make_optimizer(d):
    return SimulatedAnnealing(
        d['gather'], d['offsets'], d['dt'], vel_min=VEL_MIN, vel_max=VEL_MAX,
        min_sep_ms=MIN_SEP_MS, objective='threshold', tau=d['tau'],
        p_insert=P_INSERT, p_remove=P_REMOVE, seed=SEED, max_evals=MAX_EVALS,
        step_time=STEP_TIME, step_vel=STEP_VEL,
        record_trajectory=True, record_every=RECORD_EVERY, **SEMBLANCE,
        t0=SA_T0, p0=SA_P0, t_final_ratio=SA_FINAL_RATIO)

runs = {}
for name in CASES:
    d = data[name]
    opt = make_optimizer(d).run()
    runs[name] = opt
    t_s, v_s = opt.get_result()
    m = match_reference(t_s * d['dt'], v_s, d['reference'])
    print(f"{name:26s} f = {opt.best_score:6.3f}   K = {len(t_s)}   "
          f"on primary {m['n_on_primary']}/{m['n_primaries']}, on multiple {m['n_on_multiple']}, "
          f"F1 {m['f1']:.2f}   ({opt.exec_time_s:.0f} s, {len(opt.trajectory)} records)")

## 6. Objective and number of picks along the run

Top: objective of the current solution (blue) and best so far (red).
At the start the temperature is high and many worse moves are accepted:
the blue curve wanders far below the red one (exploration). As T drops,
fewer worse moves pass and the blue curve settles on the red one
(intensification). Bottom: number of picks K of the current solution.
One column per case.

In [ ]:
fig, axes = plt.subplots(2, len(CASES), figsize=(8 * len(CASES), 7), sharex=True, sharey='row')
for col, name in enumerate(CASES):
    search_viz.plot_score_trace(runs[name], ax=axes[0, col])
    axes[0, col].set_title(f'{ALG} — {LABEL[name]}')
    search_viz.plot_n_picks(runs[name], ax=axes[1, col])
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f'search_{TAG}_trace.png'), dpi=150)
plt.show()

## 7. Snapshots of the current solution

Six moments of each run, evenly spaced (one row per case). White circles:
current solution; black X: best solution found up to that moment (when
different).

In [ ]:
fig, axes = plt.subplots(len(CASES), 6, figsize=(20, 7 * len(CASES)), sharey=True)
for row, name in enumerate(CASES):
    d = data[name]
    search_viz.plot_snapshots(runs[name], d['panel'], d['extent'], d['dt'], n=6,
                              reference=d['reference'], axes=axes[row])
    for ax in axes[row]:
        ax.set_xlim(VEL_MIN, V_SHOW)
    axes[row, 0].set_ylabel(f'{LABEL[name]}\nTime (ms)')
plt.suptitle(f'{ALG} — current solution during the run', y=1.0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f'search_{TAG}_snapshots.png'), dpi=110, bbox_inches='tight')
plt.show()

## 8. Where the search looked

Every new pick the algorithm evaluated (a pick of an evaluated solution
that was not in the current one), as a 2D histogram over the panel (gray).
Dense regions are where the evaluations were spent; regions without color
were never tried.

In [ ]:
fig, axes = plt.subplots(1, len(CASES), figsize=(8 * len(CASES), 8))
for ax, name in zip(axes, CASES):
    d = data[name]
    search_viz.plot_proposals(runs[name], d['panel'], d['extent'], d['dt'], ax=ax,
                              reference=d['reference'])
    ax.set_title(f'{ALG} — {LABEL[name]}\n{len(runs[name].proposals)} new picks evaluated')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f'search_{TAG}_proposals.png'), dpi=130)
plt.show()

## 9. Temperature and acceptance

Top: temperature along the run (log scale). Bottom: share of the worse
neighbors that were accepted, in windows of evaluations — the practical
effect of the temperature. One column per case (T0 differs because it is
set from each case's tau).

In [ ]:
fig, axes = plt.subplots(2, len(CASES), figsize=(8 * len(CASES), 7), sharex=True)
for col, name in enumerate(CASES):
    opt  = runs[name]
    traj = [r for r in opt.trajectory if r['event'] == 'iter']
    axes[0, col].semilogy([r['n_evals'] for r in traj], [r['temperature'] for r in traj])
    axes[0, col].set_ylabel('Temperature T'); axes[0, col].grid(True, alpha=0.3, which='both')
    axes[0, col].set_title(f'{LABEL[name]}  (T0 = {opt.t0:.3f})')
    WINDOW = max(1, len(traj) // 50)
    centers, rates = [], []
    for i in range(0, len(traj) - WINDOW + 1, WINDOW):
        block = [r for r in traj[i:i + WINDOW] if r['accepted'] is not None and r['delta'] < 0]
        if block:
            centers.append(traj[i + WINDOW // 2]['n_evals'])
            rates.append(np.mean([r['accepted'] for r in block]))
    axes[1, col].plot(centers, rates, '.-')
    axes[1, col].set_ylim(0, 1); axes[1, col].set_ylabel('Worse moves accepted')
    axes[1, col].set_xlabel('Objective evaluations'); axes[1, col].grid(True, alpha=0.3)
    a = opt.accept_stats
    print(f"{name:26s} improving {a['better']}, worse accepted {a['worse_accepted']}, "
          f"worse rejected {a['worse_rejected']}; moves {opt.move_stats}")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f'search_{TAG}_temperature.png'), dpi=150)
plt.show()

## 10. Animation

One GIF per case: the current solution moving over the panel (subsampled
to at most `MAX_FRAMES` frames), saved in `results/figures/`. It can take
a minute per case.

In [ ]:
MAX_FRAMES = 200
FPS        = 10
for name in CASES:
    d = data[name]
    gif = os.path.join(FIG_DIR, f'search_{TAG}_{name}.gif')
    search_viz.animate_search(runs[name], d['panel'], d['extent'], d['dt'], gif,
                              max_frames=MAX_FRAMES, fps=FPS, reference=d['reference'],
                              title=f'{ALG} — {LABEL[name]}')
    print(f'Saved: {gif}')
    display(Image(filename=gif))

## 11. Summary

**What to look for in Simulated Annealing:**
- the objective of the current solution starts far below the best and
  noisy (high temperature: almost every move is accepted, the search
  wanders) and narrows onto the best as the temperature falls — the
  explicit exploration-to-intensification schedule;
- unlike Hill Climbing there are no restarts, and unlike Tabu Search
  there is no memory: the only thing that lets it leave a local optimum
  is the probability of accepting worse moves, which vanishes at the end;
- once cold, a missing event can only come back through an insertion
  that draws time and velocity uniformly, which rarely lands on a small
  blob;
- compare the three notebooks (Hill Climbing, Tabu Search, Simulated
  Annealing): same moves, same budget, same cases, different acceptance
  strategy.

**With vs. without free surface.** The same algorithm, seed and budget on
both cases: the differences between the two columns come from the data.
With the free surface, the strong peg-leg of top C sits below top C with a
lower velocity, and the increasing-velocity constraint makes the two
incompatible — look for picks squeezed in a narrow velocity corridor and
for top C never being visited. Without it, all four primaries are
reachable and the run should settle on them.